# Aula 6 — Qualidade de Dados e Monitoramento

**Disciplina:** Big Data Processing — MBA Engenharia de Dados (Mackenzie)

**Objetivo:** Implementar framework de qualidade com checks, quarentena e relatório.

---

## Instruções

1. Execute cada célula sequencialmente
2. Observe como checks de qualidade identificam problemas nos dados
3. Ao final, resolva o **Desafio** proposto

---

## 1. Configuração e Carga dos Dados com Problemas

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnan, isnull, lit, sum as spark_sum,
    current_timestamp, concat_ws, round, expr, countDistinct
)
from dataclasses import dataclass
from typing import List, Dict, Tuple
from datetime import datetime

spark = SparkSession.builder \
    .appName("DataFlow-Aula06-Qualidade") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"SparkSession: {spark.version}")

In [ ]:
# Carregar dados COM problemas (propositalmente sujos)
df = spark.read.parquet("/home/jovyan/work/data/aula_06/dados_sujos/vendas_problemas.parquet")

print(f"Registros carregados: {df.count():,}")
df.printSchema()
df.show(5)

## 2. Estrutura do Framework de Qualidade

In [ ]:
@dataclass
class CheckResult:
    """Resultado padronizado de um check de qualidade."""
    check_name: str
    passed: bool
    metric_value: float  # 0.0 a 1.0
    threshold: float
    severity: str  # 'critical' | 'warning' | 'info'
    details: dict

    def __str__(self):
        status = "PASSED" if self.passed else "FAILED"
        icon = "✅" if self.passed else "❌"
        return f"{icon} {self.check_name}: {status} (metric={self.metric_value:.4f}, threshold={self.threshold})"


class DataQualityFramework:
    """Framework reutilizavel de qualidade de dados."""

    def __init__(self, spark_session):
        self.spark = spark_session
        self.results: List[CheckResult] = []

    def check_completeness(self, df, columns: List[str], threshold: float = 0.95) -> CheckResult:
        """Verifica % de valores nao-nulos nas colunas."""
        total = df.count()
        details = {}
        min_completeness = 1.0

        for c in columns:
            non_null = df.filter(col(c).isNotNull()).count()
            rate = non_null / total
            details[c] = round(rate, 4)
            min_completeness = min(min_completeness, rate)

        result = CheckResult(
            check_name="completeness",
            passed=min_completeness >= threshold,
            metric_value=round(min_completeness, 4),
            threshold=threshold,
            severity="critical",
            details=details
        )
        self.results.append(result)
        return result

    def check_uniqueness(self, df, key_columns: List[str], threshold: float = 0.95) -> CheckResult:
        """Verifica % de registros unicos nas colunas-chave."""
        total = df.count()
        distinct = df.select(key_columns).distinct().count()
        uniqueness = distinct / total

        result = CheckResult(
            check_name="uniqueness",
            passed=uniqueness >= threshold,
            metric_value=round(uniqueness, 4),
            threshold=threshold,
            severity="critical",
            details={"total": total, "distinct": distinct, "duplicates": total - distinct}
        )
        self.results.append(result)
        return result

    def check_validity(self, df, rules: Dict[str, str], threshold: float = 0.90) -> CheckResult:
        """Verifica regras de dominio (expressoes SQL)."""
        total = df.count()
        combined = df
        for rule_expr in rules.values():
            combined = combined.filter(expr(rule_expr))
        valid = combined.count()
        validity = valid / total

        result = CheckResult(
            check_name="validity",
            passed=validity >= threshold,
            metric_value=round(validity, 4),
            threshold=threshold,
            severity="warning",
            details={"rules": list(rules.keys()), "valid": valid, "invalid": total - valid}
        )
        self.results.append(result)
        return result

    def generate_report(self) -> dict:
        """Gera relatorio consolidado."""
        total = len(self.results)
        passed = sum(1 for r in self.results if r.passed)
        failed = total - passed
        score = sum(r.metric_value for r in self.results) / total if total > 0 else 0

        return {
            "total_checks": total,
            "passed": passed,
            "failed": failed,
            "overall_score": round(score, 4),
            "gate_status": "PASSED" if failed == 0 else "FAILED"
        }

print("Framework de Qualidade definido!")

## 3. Executar Checks de Qualidade

In [ ]:
# Instanciar o framework
qa = DataQualityFramework(spark)

# Check 1: Completude
print("=" * 60)
print("CHECK 1: COMPLETUDE")
print("=" * 60)
result_completude = qa.check_completeness(
    df,
    columns=["order_id", "customer_id", "product_id", "order_date", "total_amount"],
    threshold=0.95
)
print(result_completude)
print(f"  Detalhes: {result_completude.details}")

In [ ]:
# Check 2: Unicidade
print("=" * 60)
print("CHECK 2: UNICIDADE")
print("=" * 60)
result_unicidade = qa.check_uniqueness(
    df,
    key_columns=["order_id"],
    threshold=0.95
)
print(result_unicidade)
print(f"  Detalhes: {result_unicidade.details}")

In [ ]:
# Check 3: Validade de dominio
print("=" * 60)
print("CHECK 3: VALIDADE DE DOMINIO")
print("=" * 60)
result_validade = qa.check_validity(
    df,
    rules={
        "quantidade_positiva": "quantity > 0",
        "preco_positivo": "unit_price > 0",
        "total_positivo": "total_amount > 0"
    },
    threshold=0.90
)
print(result_validade)
print(f"  Detalhes: {result_validade.details}")

## 4. Sistema de Quarentena

In [ ]:
# Definir regras de quarentena
quarantine_rules = {
    "order_id_nulo": col("order_id").isNotNull(),
    "customer_id_nulo": col("customer_id").isNotNull(),
    "quantidade_invalida": col("quantity") > 0,
    "preco_invalido": col("unit_price") > 0,
    "total_invalido": col("total_amount") > 0,
}

# Marcar violacoes
df_marked = df
for rule_name, rule_condition in quarantine_rules.items():
    df_marked = df_marked.withColumn(
        f"_pass_{rule_name}",
        when(rule_condition, lit(None)).otherwise(lit(rule_name))
    )

# Consolidar motivos de quarentena
flag_cols = [f"_pass_{name}" for name in quarantine_rules.keys()]
df_marked = df_marked.withColumn(
    "_quarantine_reasons",
    concat_ws(", ", *[col(c) for c in flag_cols])
)

# Separar validos e quarentena
df_valid = df_marked.filter(col("_quarantine_reasons") == "").drop(*flag_cols, "_quarantine_reasons")
df_quarantine = df_marked.filter(col("_quarantine_reasons") != "") \
    .withColumn("_quarantine_ts", current_timestamp()) \
    .withColumn("_severity", lit("critical")) \
    .drop(*flag_cols)

# Metricas
total_original = df.count()
total_valid = df_valid.count()
total_quarantine = df_quarantine.count()

print(f"Total original:   {total_original:,}")
print(f"Validos:          {total_valid:,}")
print(f"Quarentena:       {total_quarantine:,}")
print(f"Taxa quarentena:  {total_quarantine/total_original*100:.1f}%")
print(f"\nConservacao: {total_valid + total_quarantine} == {total_original} -> {total_valid + total_quarantine == total_original}")

In [ ]:
# Amostra da quarentena
print("=== REGISTROS EM QUARENTENA ===")
df_quarantine.select("order_id", "total_amount", "quantity", "_quarantine_reasons").show(10, truncate=False)

In [ ]:
# Breakdown por motivo
print("=== QUARENTENA POR MOTIVO ===")
from pyspark.sql.functions import explode, split

df_quarantine \
    .withColumn("motivo", explode(split(col("_quarantine_reasons"), ", "))) \
    .groupBy("motivo") \
    .agg(count("*").alias("registros")) \
    .orderBy("registros", ascending=False) \
    .show()

## 5. Relatório Consolidado

In [ ]:
# Gerar relatorio
report = qa.generate_report()

print("=" * 60)
print("RELATORIO DE QUALIDADE")
print("=" * 60)
print(f"  Checks executados:  {report['total_checks']}")
print(f"  Passed:             {report['passed']}")
print(f"  Failed:             {report['failed']}")
print(f"  Score geral:        {report['overall_score']*100:.1f}%")
print(f"  Quality Gate:       {report['gate_status']}")
print("=" * 60)
print(f"\n  Registros validos:  {total_valid:,} ({total_valid/total_original*100:.1f}%)")
print(f"  Em quarentena:      {total_quarantine:,} ({total_quarantine/total_original*100:.1f}%)")
print("\nChecks individuais:")
for r in qa.results:
    print(f"  {r}")

In [ ]:
spark.stop()
print("SparkSession encerrada.")

---

# DESAFIO

## Adicionar Checks de Integridade Referencial e Alertas

Usando o framework acima como base, implemente:

### Requisitos:

1. **Check de integridade referencial**: Carregue `vendas_referencia.parquet` como tabela de referência. Verifique se todos os `customer_id` das vendas existem na referência usando `left_anti` join. Reporte % de órfãos.

2. **Adicione o método `check_referential_integrity`** à classe `DataQualityFramework`:
   ```python
   def check_referential_integrity(self, df, ref_df, join_key, threshold=0.95):
       # orphans = df.join(ref_df, on=join_key, how="left_anti")
       ...
   ```

3. **Implemente um quality gate** que:
   - Se algum check `critical` falhar → pipeline PARA (raise Exception)
   - Se apenas checks `warning` falharem → pipeline continua com alerta

4. **Adicione check de freshness**: verifique se a `order_date` mais recente não é anterior a 7 dias (dados desatualizados)

5. **Salve o relatório** como DataFrame e persista em Parquet para histórico de execuções

### Dicas:
- `df.join(ref_df.select(join_key), on=join_key, how="left_anti")` retorna apenas registros sem match
- `df.agg(max(\"order_date\")).collect()[0][0]` para obter a data mais recente
- `datediff(current_date(), max_date)` para calcular freshness

### Bonus:
- Implemente trending: compare score atual com execução anterior (historico)
- Adicione thresholds diferenciados por coluna (ex: order_id=99%, product_id=95%)
- Crie um DataFrame de métricas formatado como tabela para exibição

---

**Boa sorte!** Quality checks são 15% da nota do Projeto Final.

In [ ]:
# ===========================================================
# SEU CODIGO DO DESAFIO AQUI
# ===========================================================

